In [ ]:
# SINGLE CELL: LiaHR (Label-in-a-Haystack Rectification) wrapper for PSI/PSR labels
# - Reads from: messages.json
# - Works on a COPY: 24_LiaHR.json (created once if missing)
# - For each message with candidate labels: runs LiaHR verify/rectify
# - Writes to disk AFTER EVERY MESSAGE (max resumability)
# - Exports: 24_LiaHR.xlsx
# - Tracks: total elapsed time across resumes, total input/output/cached tokens, and estimated cost (USD)

import os, json, time, re, sys, shutil, uuid, random
from datetime import datetime
from typing import Dict, Any, List
from openai import OpenAI

# -----------------------------
# 0) Install/import deps
# -----------------------------
def _pip_install(pkgs: List[str]):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import pandas as pd
except Exception:
    _pip_install(["pandas", "openpyxl"])
    import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    _pip_install(["python-dotenv"])
    from dotenv import load_dotenv

try:
    import tiktoken
except Exception:
    _pip_install(["tiktoken"])
    import tiktoken

try:
    from openai import OpenAI
except Exception:
    _pip_install(["openai"])
    from openai import OpenAI

# -----------------------------
# 1) Config
# -----------------------------
ROOT_DIR = os.getcwd()

SOURCE_MESSAGES_PATH = os.path.join(ROOT_DIR, "LiaHRInput.json")
RUN_BASENAME         = "24_LiaHR"
WORK_MESSAGES_PATH   = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.json")
XLSX_OUT             = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.xlsx")
STATS_PATH           = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_run_stats.json")
DEMO_POOL_PATH       = os.path.join(ROOT_DIR, "LiaHRDemoPool.json")

ANNOTATION_MODEL     = "gpt-5.1"
MAX_OUTPUT_TOKENS    = 800  # LiaHR outputs are small

# LiaHR shots (additional demos besides the query-as-demo)
N_SHOTS              = 8
RANDOM_SEED          = 7
random.seed(RANDOM_SEED)

# Rectification policy:
# - If True: overwrite labels when mismatch (LiaHR "rectification")
# - If False: only flag mismatch, keep original labels
DO_RECTIFY           = True

# Pricing (per 1M tokens) — adjust if your contract differs
PRICE_INPUT_PER_1M   = 1.25
PRICE_CACHED_PER_1M  = 0.125
PRICE_OUTPUT_PER_1M  = 10.00

BINARY_FIELDS = [
    "DirectAddress", "Attachment", "SelfDisclose", "ReplySeeking",
    "CommRitual", "Monetary", "Backseat", "Emotes"
]
OTHER_FIELDS = ["Dimension"]

# -----------------------------
# 2) Developer prompt (you can keep your full rulebook)
#    Important: LiaHR relies on "do the task" instructions, not "copy labels".
# -----------------------------
DEVELOPER_PROMPT = r"""
You are an annotation model. Your task is to label each Twitch chat message with a set of binary/ternary fields using the provided schema.
Follow definitions and decision rules exactly.

Return ONLY a single JSON OBJECT (no wrapper object, no arrays, no text) with keys:
DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension, Notes.

Each label field must be one of:
"Y" = present
""  = not present / leave blank

For Dimension, use exactly one of:
"P" if positive/affiliative stance toward streamer/community
"N" if negative/hostile stance toward streamer/community
No neutral/mixed category. If mixed, label based on the dominant intent (“punchline”).

WHAT YOU ARE ANNOTATING
Annotate message-level expressed cues in livestream chat from Twitch. Multi-label is allowed.
Do not infer hidden intent beyond the text and immediate conversational form.

CATEGORY DECISION RULES
A) Dimension (valence)
Set Dimension independently from other categories.
P (Positive): praise/support, gratitude, affection, encouragement, admiration; playful teasing that is clearly supportive.
N (Negative): insults, hostility, shaming, threats, aggressive blame; hate-watch enjoyment of failure.
Mixed: choose dominant intent.

B) PSI/PSR-leaning categories (multi-label)
1) Direct Address / second-person to streamer -> DirectAddress
Mark "Y" only when the message targets the streamer as an addressee:
- explicit @streamer
- streamer name used as addressee (“Emiru, …”)
- second-person pronouns clearly referring to streamer not another person (“you/your/u”) with directed statement/command
Avoid false positives:
- talking about streamer: “emiru is cracked”
- addressing chat: “you guys…”
- ambiguous “you” -> blank unless strong cue (name/@mention).

Important Point! You should keep an eye on morphs, euphemisms, and varying forms of chatting most common for Gen-Z, like "ya", "yo" etc.

2) Attachment / affection / relational warmth -> Attachment
Mark "Y" for warmth/care/pride/missing/love toward streamer or relational closeness markers.
Attachment-strengtheners (no extra labels; just annotate Attachment when applicable):
- longitudinal persistence beyond exposure: off-stream anticipation, absence felt, “all week,” “can’t wait,” “glad you’re back”
- investment/loyalty talk: “watch every stream,” “since 2018,” “never miss a VOD,” schedule organized around streams
- social realism / real-life tie framing: “you’re like family,” “real friend,” “like a brother/sister”
- anthropomorphism/personification: “you care about us,” “she understands me,” “he worries about chat”
Avoid false positives:
- “love this game” (not streamer-directed)
- generic hype alone (“Pog”, “Lets go”) unless clearly relational.

3) Identification / self-disclosure / homophily -> SelfDisclose
Mark "Y" when viewer shares personal info or similarity claims aimed at relational connection:
- “I’m also…”, “same here…”, “as a fellow…”, "I also do that..."
- personal disclosure positioned as sharing-with-streamer (“I had a rough day, thanks for being here”)
- “I have an exam tomorrow, wish me luck” can count if framed as a bid for connection.
Strengthener (no new label):
- knowledge/narrative recall is not SelfDisclose by itself; use it to support DirectAddress/Attachment/ReplySeeking if it functions that way.

4) Reciprocity bids / reply-seeking -> ReplySeeking
Mark "Y" when seeking acknowledgment/response/recognition:
- “notice me,” “read this,” “answer please,” “remember me,” “did you see my last message,” “thanks for replying last week,” "why you not seeing me"
- Or asking them to do something ordinary like drinking water or blink for safety of eyes. 
Avoid false positives:
- generic info questions not aimed at streamer (“what game is this?”) unless streamer-directed by name/@mention. 

5) Community ritual talk & co-creation -> CommRitual
Mark "Y" for shared rituals/inside jokes/coordinated norms tied to community:
- “spam LUL,” “copy pasta time,” “chat do the thing,” “only real ones remember…,” “as always…,” "Let's go..."
Avoid false positives:
- emotes alone ≠ ritual talk without ritual framing.

6) Monetary support actions -> Monetary
Mark "Y" for subs/donos/gifts/streaks or accompanying text:
- “subbed X months,” “gifted subs,” “dono,” “renewed,” “superchat”
Hard rule: If the message tag is "USERNOTICE" instead of "PRIVATEMESSAGE" ALWAYS mark Monetary="Y".
Avoid false positives:
- unrelated money talk (“costs $60”).

ADDED CATEGORIES (behavioral / surface form)
7) Backseat guidance -> Backseat
Mark "Y" for gameplay/stream/content direction/coaching/strategy/idea:
- “go left,” “use ult,” “buy armor,” “skip cutscene,” “watch out…,” "You have to change it in the menu"
Or general questions asking regarding the topic:
- "Hassan are they capitalists?", "you should do a comparison on their technical data"
Backseat can co-occur with DirectAddress, but backseating is not parasocial by itself.

8) Emotes/emoji presence -> Emotes
Mark "Y" if any Twitch-style emote tokens or unicode emojis appear, including <3, and tokens like Kappa, pogchamp, KEKW, etc.

CONTRASTIVE DEMONSTRATIONS (BEHAVIORAL TARGETS)
Follow the demos for common pitfalls:
Demo A — DirectAddress vs about-talk; “you” ambiguity
General: emiru is the streamer; “chat” = audience. Local: streamer just returned; greetings happening.
Inputs:
(1) "Everyone missed you beautiful Emiru <3 <3 WutFace"
(2) "emiru is cracked today"
(3) "you guys missed her too right? lol"
Correct:
(1) DirectAddress=Y (name + “you”), Attachment=Y (“missed you”, warmth), Emotes=Y, Dimension=P
(2) About-talk only: DirectAddress blank; Dimension=P (praise)
(3) Targets chat (“you guys”): DirectAddress blank; Dimension=P (light/affiliative)
Avoid:
- Don’t set DirectAddress just because streamer name appears (2)
- Don’t set DirectAddress from “you” when it clearly addresses chat (3)

Demo B — Loyalty/investment as Attachment; recall ≠ SelfDisclose; mixed valence punchline rule
General: hasanabi political streams; “since YEAR” indicates tenure.
Inputs:
(10) "Been watching since 2019. Never miss a VOD."
(11) "Remember that 2019 meltdown? downhill ever since lol"
(12) "I love you but you’re being an idiot tonight"
Correct:
(10) Attachment=Y (investment/loyalty), SelfDisclose=Y, Dimension=P
(11) Dimension=N (criticism); SelfDisclose blank (recall is not personal disclosure)
(12) Attachment=Y (“love you”), Dimension=N (dominant thrust is insult/punchline)
Avoid:
- Don’t force SelfDisclose from narrative recall (11)
- Don’t label Dimension=P just because affection phrase exists (12)

Demo C — USERNOTICE monetary override + co-occurrence
Rule: msgTag="USERNOTICE" => Monetary=Y ALWAYS.
Input:
(20) msgTag=USERNOTICE, "Renewed my sub for 24 months — love you Emiru <3"
Correct:
Monetary=Y (forced), Attachment=Y, DirectAddress=Y (name used as addressee), Emotes=Y (<3), Dimension=P
Avoid:
- Never leave Monetary blank on USERNOTICE even if message is mostly affection

Return ONLY one JSON object (not an array), no code fences, no extra text.
"""

# -----------------------------
# 3) Env + client
# -----------------------------
load_dotenv(os.path.join(ROOT_DIR, ".env"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists with OPENAI_API_KEY=...")

client = OpenAI()

# -----------------------------
# 4) Helpers
# -----------------------------
def now_ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def safe_write_json(path: str, obj: Any, retries: int = 10, sleep: float = 0.05):
    directory = os.path.dirname(path)
    os.makedirs(directory, exist_ok=True)
    tmp = f"{path}.{uuid.uuid4().hex}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    for _ in range(retries):
        try:
            shutil.move(tmp, path)
            return
        except PermissionError:
            time.sleep(sleep)
    raise PermissionError(f"Could not replace {path} after {retries} retries")

def load_json_if_exists(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def ensure_working_copy():
    if not os.path.exists(SOURCE_MESSAGES_PATH):
        raise FileNotFoundError(f"messages.json not found at: {SOURCE_MESSAGES_PATH}")
    if not os.path.exists(WORK_MESSAGES_PATH):
        shutil.copy2(SOURCE_MESSAGES_PATH, WORK_MESSAGES_PATH)
        print(f"[{now_ts()}] Created working copy: {WORK_MESSAGES_PATH}")
    else:
        print(f"[{now_ts()}] Using existing working copy (resume): {WORK_MESSAGES_PATH}")

def load_messages() -> List[Dict[str, Any]]:
    with open(WORK_MESSAGES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Working messages file must be a JSON array of message objects.")
    return data

def ensure_fields(msg: Dict[str, Any]):
    for k in BINARY_FIELDS + OTHER_FIELDS:
        msg.setdefault(k, "")
    for k in ["raw_msg", "msgTag", "streamer", "Annotated", "Notes"]:
        msg.setdefault(k, "")
    # LiaHR bookkeeping
    msg.setdefault("LiaHR_Done", "False")
    msg.setdefault("LiaHR_Status", "")      # copied / replaced / no_candidate / error
    msg.setdefault("LiaHR_MismatchFields", [])
    msg.setdefault("LiaHR_LatencySec", 0.0)

def is_liahr_done(msg: Dict[str, Any]) -> bool:
    return str(msg.get("LiaHR_Done", "")).strip().lower() == "true"

def mark_liahr_done(msg: Dict[str, Any]):
    msg["LiaHR_Done"] = "true"

def enforce_binary(v: Any) -> str:
    return "Y" if str(v).strip().upper() == "Y" else ""

def enforce_dimension(v: Any, fallback: str = "P") -> str:
    s = str(v).strip().upper()
    if s in ("P", "POS", "POSITIVE"):
        return "P"
    if s in ("N", "NEG", "NEGATIVE"):
        return "N"
    return fallback

def clip_notes(text: str, max_words: int = 20) -> str:
    t = re.sub(r"\s+", " ", (text or "").strip())
    if not t:
        return ""
    words = t.split(" ")
    return t if len(words) <= max_words else " ".join(words[:max_words]).rstrip() + "…"

def get_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-5.1")
    except Exception:
        return tiktoken.get_encoding("o200k_base")

ENC = get_encoding()

def tok_len(s: str) -> int:
    return len(ENC.encode(s or ""))

def usage_breakdown(resp) -> Dict[str, Any]:
    u = getattr(resp, "usage", None)
    if not u:
        return {}
    try:
        return u if isinstance(u, dict) else u.model_dump()
    except Exception:
        try:
            return dict(u)
        except Exception:
            return {"raw_usage": str(u)}

def estimate_cost_usd(total_input: int, total_cached: int, total_output: int) -> float:
    cached = max(0, int(total_cached or 0))
    inp = max(0, int(total_input or 0))
    non_cached = max(0, inp - cached)
    cost = 0.0
    cost += (non_cached / 1_000_000.0) * PRICE_INPUT_PER_1M
    cost += (cached     / 1_000_000.0) * PRICE_CACHED_PER_1M
    cost += (max(0, int(total_output or 0)) / 1_000_000.0) * PRICE_OUTPUT_PER_1M
    return cost

def load_stats() -> Dict[str, Any]:
    stats = load_json_if_exists(STATS_PATH, default={})
    if not isinstance(stats, dict):
        stats = {}
    stats.setdefault("accumulated_seconds", 0.0)
    stats.setdefault("runs", 0)
    stats.setdefault("total_input_tokens", 0)
    stats.setdefault("total_output_tokens", 0)
    stats.setdefault("total_cached_tokens", 0)
    stats.setdefault("last_run_started_at", "")
    stats.setdefault("last_run_ended_at", "")
    # LiaHR counts
    stats.setdefault("liahr_copied", 0)
    stats.setdefault("liahr_replaced", 0)
    stats.setdefault("liahr_no_candidate", 0)
    stats.setdefault("liahr_errors", 0)
    return stats

def save_stats(stats: Dict[str, Any]):
    safe_write_json(STATS_PATH, stats)

def short_preview(s: str, n: int = 90) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    return s if len(s) <= n else s[:n-1] + "…"

def call_with_retries(kwargs, tries=6, base_sleep=1.5):
    last_err = None
    for attempt in range(1, tries + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as e:
            last_err = e
            sleep = base_sleep * (2 ** (attempt - 1))
            print(f"[{now_ts()}]   ERROR (attempt {attempt}/{tries}): {e}. Sleeping {sleep:.1f}s then retry...")
            time.sleep(sleep)
    raise last_err

def enforce_dimension_strict(v: Any) -> str:
    s = str(v).strip().upper()
    if s in ("P", "N"):
        return s
    raise ValueError(f"Invalid Dimension: {v!r}")


def get_output_text(resp) -> str:
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    out = getattr(resp, "output", None) or []
    for item in out:
        if isinstance(item, dict) and item.get("type") == "message":
            for c in (item.get("content") or []):
                if isinstance(c, dict) and c.get("type") in ("output_text", "text"):
                    return c.get("text", "") or ""
    return ""

def try_repair_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    t = t.strip()
    m_obj = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if m_obj:
        t = m_obj.group(0)
    t = t.replace("\t", " ")
    t = re.sub(r",\s*([}\]])", r"\1", t)
    return t.strip()

def reformat_to_json_only(bad_text: str) -> str:
    fix_prompt = (
        "You will be given text that is intended to be JSON but is invalid.\n"
        "Rewrite it into VALID JSON ONLY.\n"
        "Do not change meanings or labels. Do not omit any keys.\n"
        "Output ONLY valid JSON.\n\n"
        "TEXT:\n" + bad_text
    )
    r = client.responses.create(
        model=ANNOTATION_MODEL,
        input=[{"role": "user", "content": fix_prompt}],
        max_output_tokens=400,
        temperature=0.0,
    )
    return (get_output_text(r) or "").strip()

def parse_single_json(resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")
    t = try_repair_json(text)

    def _loads_or_none(s: str):
        try:
            return json.loads(s)
        except Exception:
            return None

    obj = _loads_or_none(t)
    if obj is None:
        fixed = reformat_to_json_only(text)
        fixed2 = try_repair_json(fixed)
        obj = _loads_or_none(fixed2)
        if obj is None:
            raise RuntimeError(f"JSON parse failed.\nRAW:\n{text[:1500]}\n\nFIXED:\n{fixed[:1500]}")

    if not isinstance(obj, dict):
        raise RuntimeError(f"Expected JSON object, got {type(obj)}. text={text[:500]}")

    for k in (BINARY_FIELDS + OTHER_FIELDS + ["Notes"]):
        obj.setdefault(k, "")

    return obj

def has_candidate_labels(msg: Dict[str, Any]) -> bool:
    if str(msg.get("Annotated","")).strip().lower() != "true":
        return False
    return str(msg.get("Dimension","")).strip().upper() in ("P","N")

def pack_candidate_labels(msg: Dict[str, Any]) -> Dict[str, str]:
    labels = {f: enforce_binary(msg.get(f, "")) for f in BINARY_FIELDS}
    if str(msg.get("msgTag","")).strip().upper() == "USERNOTICE":
        labels["Monetary"] = "Y"

    dim = str(msg.get("Dimension","")).strip().upper()
    if dim not in ("P","N"):
        raise ValueError("Candidate labels missing/invalid Dimension although Annotated=True.")
    labels["Dimension"] = dim
    return labels

def pack_pred_labels(out_obj: Dict[str, Any], msg: Dict[str, Any]) -> Dict[str, str]:
    labels = {f: enforce_binary(out_obj.get(f, "")) for f in BINARY_FIELDS}

    if str(msg.get("msgTag", "")).strip().upper() == "USERNOTICE":
        labels["Monetary"] = "Y"

    labels["Dimension"] = enforce_dimension_strict(out_obj.get("Dimension", ""))
    return labels

def build_liahr_user_prompt(query_payload, query_candidate_labels, demo_pairs):
    demos = demo_pairs.copy()
    insert_pos = random.randint(0, len(demos))
    demos.insert(insert_pos, {"input": query_payload, "labels": query_candidate_labels})

    return (
        "Below is a JSON array of labeled examples for the task. "
        "Some labels may be noisy. Perform the task; do NOT mechanically copy labels.\n\n"
        "LABELED_EXAMPLES_JSON:\n"
        + json.dumps(
            [{"INPUT_MESSAGE": d["input"], "LABELS": d["labels"]} for d in demos],
            ensure_ascii=False
        )
        + "\n\n"
        "Now label the following QUERY message from scratch.\n"
        "Return ONLY one JSON object with keys: "
        + ", ".join(BINARY_FIELDS + ["Dimension", "Notes"]) + ".\n\n"
        "QUERY:\n"
        + json.dumps({"INPUT_MESSAGE": query_payload}, ensure_ascii=False)
    )

def choose_demo_pool(demo_pool: List[Dict[str, Any]],
                     k: int,
                     idx_seed: int,
                     query_payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    rng = random.Random(RANDOM_SEED + idx_seed)

    q_tag = (query_payload.get("msgTag") or "").strip()
    q_raw = (query_payload.get("raw_msg") or "").strip()

    candidates = []
    for j, m in enumerate(demo_pool):
        if not has_candidate_labels(m):
            continue
        raw = (m.get("raw_msg") or "").strip()
        tag = (m.get("msgTag") or "").strip()
        if not raw:
            continue
        # avoid accidental exact-duplicate leakage
        if tag == q_tag and raw == q_raw:
            continue
        candidates.append(j)

    rng.shuffle(candidates)
    pick = candidates[:k]

    out = []
    for j in pick:
        m = demo_pool[j]
        out.append({
            "input": {"msgTag": m.get("msgTag", ""), "raw_msg": m.get("raw_msg", "")},
            "labels": pack_candidate_labels(m),
        })
    return out

# -----------------------------
# 5) Setup working copy + load data + stats
# -----------------------------
ensure_working_copy()

def load_demo_pool() -> List[Dict[str, Any]]:
    if not os.path.exists(DEMO_POOL_PATH):
        raise FileNotFoundError(f"Demo pool not found: {DEMO_POOL_PATH}")
    with open(DEMO_POOL_PATH, "r", encoding="utf-8") as f:
        dp = json.load(f)
    if not isinstance(dp, list):
        raise ValueError("Demo pool must be a JSON array of message objects.")
    # only need the fields used by has_candidate_labels/pack_candidate_labels
    for m in dp:
        for k in ["raw_msg", "msgTag", "Annotated", "Dimension"] + BINARY_FIELDS:
            m.setdefault(k, "")
    return dp

demo_pool = load_demo_pool()
print(f"[{now_ts()}] Loaded demo pool: {len(demo_pool)} items")

messages = load_messages()
for m in messages:
    ensure_fields(m)

total = len(messages)
remaining = sum(1 for m in messages if (not is_liahr_done(m)) and has_candidate_labels(m))
print(f"[{now_ts()}] Loaded {total} messages. Remaining LiaHR-eligible (has candidate labels, not done): {remaining}")

stats = load_stats()
stats["runs"] += 1
stats["last_run_started_at"] = now_ts()
save_stats(stats)

run_start = time.time()

# -----------------------------
# 6) LiaHR main loop
# -----------------------------
processed_this_run = 0

for idx, msg in enumerate(messages):
    if is_liahr_done(msg):
        continue

    raw_msg = msg.get("raw_msg", "") or ""
    msgTag  = msg.get("msgTag", "") or ""

    if not has_candidate_labels(msg):
        msg["LiaHR_Status"] = "no_candidate"
        msg["LiaHR_MismatchFields"] = []
        msg["LiaHR_LatencySec"] = 0.0
        mark_liahr_done(msg)

        stats["liahr_no_candidate"] += 1
        safe_write_json(WORK_MESSAGES_PATH, messages)

        elapsed_segment = time.time() - run_start
        stats["accumulated_seconds"] += elapsed_segment
        run_start = time.time()
        stats["last_run_ended_at"] = now_ts()
        save_stats(stats)

        continue

    query_payload = {"msgTag": msgTag, "raw_msg": raw_msg}
    cand_labels   = pack_candidate_labels(msg)
    demos         = choose_demo_pool(demo_pool, k=N_SHOTS, idx_seed=idx, query_payload=query_payload)
    user_prompt   = build_liahr_user_prompt(query_payload, cand_labels, demos)

    kwargs = dict(
        model=ANNOTATION_MODEL,
        input=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        max_output_tokens=MAX_OUTPUT_TOKENS,
        reasoning={"effort": "low"},
        # temperature=0.0,  # optional determinism
    )

    t0 = time.time()
    try:
        resp = call_with_retries(kwargs)
    except Exception as e:
        msg["LiaHR_Status"] = "error"
        msg["LiaHR_MismatchFields"] = []
        msg["LiaHR_LatencySec"] = 0.0
        mark_liahr_done(msg)

        stats["liahr_errors"] += 1
        safe_write_json(WORK_MESSAGES_PATH, messages)
        elapsed_segment = time.time() - run_start
        stats["accumulated_seconds"] += elapsed_segment
        run_start = time.time()
        stats["last_run_ended_at"] = now_ts()
        save_stats(stats)

        print(f"[{now_ts()}] #{idx} ERROR calling model: {e}")
        continue

    latency = time.time() - t0

    ud = usage_breakdown(resp)
    in_tok  = int(ud.get("input_tokens") or 0)
    out_tok = int(ud.get("output_tokens") or 0)
    cached_tok = 0
    try:
        cached_tok = int((ud.get("input_tokens_details") or {}).get("cached_tokens") or 0)
    except Exception:
        cached_tok = 0

    stats["total_input_tokens"]  += in_tok
    stats["total_output_tokens"] += out_tok
    stats["total_cached_tokens"] += cached_tok

    out_obj = parse_single_json(resp)
    try:
        pred_labels = pack_pred_labels(out_obj, msg)
    except ValueError:
        # One-shot repair: force Dimension to be P or N while keeping the rest consistent
        repair_prompt = (
            "You will be given a JSON object that MUST match the schema.\n"
            "Fix it so that:\n"
            "- Dimension is exactly 'P' or 'N' (no other values, not blank)\n"
            "- Keep other labels the same unless they are missing/invalid\n"
            "- Return ONLY the corrected JSON object\n\n"
            "BAD_JSON:\n" + json.dumps(out_obj, ensure_ascii=False)
        )

        repair_resp = call_with_retries(dict(
            model=ANNOTATION_MODEL,
            input=[{"role": "user", "content": repair_prompt}],
            max_output_tokens=200,
            temperature=0.0,
        ))

        # token accounting for repair
        ud2 = usage_breakdown(repair_resp)
        in2  = int(ud2.get("input_tokens") or 0)
        out2 = int(ud2.get("output_tokens") or 0)
        cached2 = 0
        try:
            cached2 = int((ud2.get("input_tokens_details") or {}).get("cached_tokens") or 0)
        except Exception:
            cached2 = 0

        stats["total_input_tokens"]  += in2
        stats["total_output_tokens"] += out2
        stats["total_cached_tokens"] += cached2

        out_obj = parse_single_json(repair_resp)
        pred_labels = pack_pred_labels(out_obj, msg)
    pred_labels = pack_pred_labels(out_obj, msg)
    pred_notes = clip_notes(out_obj.get("Notes", ""), max_words=20)

    mismatch_fields = [k for k in (BINARY_FIELDS + ["Dimension"]) if pred_labels.get(k, "") != cand_labels.get(k, "")]
    copied = (len(mismatch_fields) == 0)

    if copied:
        msg["LiaHR_Status"] = "copied"
        msg["LiaHR_MismatchFields"] = []
        stats["liahr_copied"] += 1
    else:
        msg["LiaHR_Status"] = "replaced" if DO_RECTIFY else "flagged"
        msg["LiaHR_MismatchFields"] = mismatch_fields
        if DO_RECTIFY:
            # Store originals
            for k in (BINARY_FIELDS + ["Dimension"]):
                msg[f"{k}_orig"] = msg.get(k, "")
            # Overwrite with LiaHR prediction
            for k in BINARY_FIELDS:
                msg[k] = pred_labels[k]
            msg["Dimension"] = pred_labels["Dimension"]
            stats["liahr_replaced"] += 1

    msg["Notes"] = pred_notes
    msg["LiaHR_LatencySec"] = float(f"{latency:.4f}")
    mark_liahr_done(msg)
    processed_this_run += 1

    # Write JSON to disk AFTER EVERY MESSAGE
    safe_write_json(WORK_MESSAGES_PATH, messages)

    # Update resumable time AFTER EVERY MESSAGE
    elapsed_segment = time.time() - run_start
    stats["accumulated_seconds"] += elapsed_segment
    run_start = time.time()
    stats["last_run_ended_at"] = now_ts()
    save_stats(stats)

    cost_now = estimate_cost_usd(stats["total_input_tokens"], stats["total_cached_tokens"], stats["total_output_tokens"])

    print(
        f"[{now_ts()}] #{idx} tag='{msgTag.strip()}' msg='{short_preview(raw_msg, 110)}'\n"
        f"  LiaHR: status={msg['LiaHR_Status']} mismatches={msg['LiaHR_MismatchFields']}\n"
        f"  usage: in={in_tok} out={out_tok} cached_in={cached_tok} | latency={latency:.2f}s\n"
        f"  totals: copied={stats['liahr_copied']} replaced={stats['liahr_replaced']} "
        f"no_candidate={stats['liahr_no_candidate']} errors={stats['liahr_errors']} | est_cost=${cost_now:.6f}\n"
    )

# -----------------------------
# 7) Export XLSX
# -----------------------------
df = pd.DataFrame(messages)

front = [
    "streamer", "msgTag", "raw_msg",
    "Annotated",
    "LiaHR_Done", "LiaHR_Status", "LiaHR_MismatchFields", "LiaHR_LatencySec",
] + BINARY_FIELDS + ["Dimension", "Notes"]

# include *_orig if present
orig_cols = [c for c in df.columns if c.endswith("_orig")]
cols = front + orig_cols + [c for c in df.columns if c not in set(front + orig_cols)]
df = df[cols]
df.to_excel(XLSX_OUT, index=False)

# -----------------------------
# 8) Final summary
# -----------------------------
remaining = sum(1 for m in messages if (not is_liahr_done(m)) and has_candidate_labels(m))

total_input  = stats["total_input_tokens"]
total_output = stats["total_output_tokens"]
total_cached = stats["total_cached_tokens"]
total_cost = estimate_cost_usd(total_input, total_cached, total_output)

print(f"\n[{now_ts()}] DONE.")
print(f"[{now_ts()}] Total messages={total} | Processed this run={processed_this_run} | Remaining LiaHR-eligible={remaining}")
print(f"[{now_ts()}] Working JSON: {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] XLSX: {XLSX_OUT}")
print(f"[{now_ts()}] Resumable time (accumulated): {stats['accumulated_seconds']:.2f} seconds")
print(f"[{now_ts()}] Tokens total: input={total_input}, cached_input={total_cached}, output={total_output}")
print(f"[{now_ts()}] Estimated total cost: ${total_cost:.6f}")
print(f"[{now_ts()}] Run stats file: {STATS_PATH}")
